# Esqueleto del modelo MILP para Keccak reducido

Este notebook construye y valida la estructura inicial del modelo de Programación Lineal Entera Mixta para versiones reducidas de Keccak.

En esta etapa se implementan:

- las variables binarias de los estados de frontera;
- la restricción de entrada diferencial no nula;
- una función objetivo provisional;
- la resolución mediante CBC;
- la inspección de variables declaradas y conectadas;
- la comparación de las configuraciones experimentales.

Todavía no se incorporan las transformaciones:

- $\theta$;
- $\rho$;
- $\pi$;
- $\chi$.

El objetivo de esta etapa es comprobar que la arquitectura del modelo funciona correctamente antes de agregar las restricciones criptográficas.

## 1. Representación de los estados de frontera

El estado reducido de Keccak se representa mediante:

$$
A^{(r)}[x,y,k],
$$

donde:

$$
x,y\in\{0,1,2,3,4\},
$$

y:

$$
k\in\{0,\ldots,z-1\}.
$$

El índice $r$ identifica un estado de frontera entre rondas.

Para un experimento con $R$ rondas se necesitan:

$$
R+1
$$

estados:

- $A^{(0)}$: entrada de la primera ronda;
- $A^{(1)}$: salida de la primera ronda;
- $A^{(2)}$: salida de la segunda ronda;
- $A^{(R)}$: salida de la última ronda.

La cantidad inicial de variables binarias de estado es:

$$
N_{\mathrm{estado}}
=
25z(R+1).
$$

## 2. Configuración de las rutas

El paquete local `keccak_milp` fue instalado en modo editable. Por tanto, debería poder importarse directamente desde el entorno virtual.

De todas formas, se mantiene una validación de las rutas para asegurar que el notebook se está ejecutando desde la estructura correcta del proyecto.

In [1]:
# ============================================================
# CONFIGURACIÓN DE RUTAS
# ============================================================

from pathlib import Path
import sys


CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

SRC_DIR = PROJECT_ROOT / "src"
RESULTS_DIR = PROJECT_ROOT / "results"
TABLES_DIR = RESULTS_DIR / "tables"

TABLES_DIR.mkdir(parents=True, exist_ok=True)


print("=" * 70)
print("RUTAS DEL PROYECTO")
print("=" * 70)
print(f"Directorio actual : {CURRENT_DIR}")
print(f"Raíz del proyecto : {PROJECT_ROOT}")
print(f"Carpeta src       : {SRC_DIR}")
print(f"Carpeta results   : {RESULTS_DIR}")
print(f"Python activo     : {sys.executable}")
print("=" * 70)

RUTAS DEL PROYECTO
Directorio actual : d:\Documentos\000. MSC\3er Ciclo\Cripto\PracticaCalificada\notebooks
Raíz del proyecto : d:\Documentos\000. MSC\3er Ciclo\Cripto\PracticaCalificada
Carpeta src       : d:\Documentos\000. MSC\3er Ciclo\Cripto\PracticaCalificada\src
Carpeta results   : d:\Documentos\000. MSC\3er Ciclo\Cripto\PracticaCalificada\results
Python activo     : d:\venvs\keccak-milp\Scripts\python.exe


## 3. Importación de dependencias

Se importan:

- `Pandas`, para organizar los resultados;
- `PuLP`, para inspeccionar el problema MILP;
- `ExperimentConfig`, para definir los escenarios;
- `KeccakMILPModel`, para construir el modelo;
- `ModelStatistics`, para consultar sus dimensiones.

In [2]:
# ============================================================
# IMPORTACIONES
# ============================================================

import pandas as pd
import pulp

from keccak_milp import (
    ExperimentConfig,
    KeccakMILPModel,
    ModelStatistics,
)
from keccak_milp.solver import available_solvers


print("=" * 70)
print("ENTORNO MILP")
print("=" * 70)
print(f"PuLP               : {pulp.__version__}")
print(f"Solvers disponibles: {available_solvers()}")
print("=" * 70)

ENTORNO MILP
PuLP               : 3.3.2
Solvers disponibles: ['PULP_CBC_CMD', 'COIN_CMD']


## 4. Construcción de un modelo inicial

Se comienza con el escenario más pequeño del trabajo:

$$
z=4,
$$

$$
R=1.
$$

El estado tiene:

$$
25z=25(4)=100
$$

bits.

Al existir un estado de entrada y un estado de salida, la cantidad total de variables declaradas es:

$$
25z(R+1)
=
25(4)(1+1)
=
200.
$$

En esta etapa estas variables representan indicadores binarios de actividad:

$$
A^{(r)}[x,y,k]\in\{0,1\}.
$$

In [3]:
# ============================================================
# CONFIGURACIÓN DEL MODELO INICIAL
# ============================================================

config_inicial = ExperimentConfig(
    z=4,
    rounds=1,
    solver="cbc",
    time_limit_seconds=60,
    mip_gap=0.0,
    verbose=False,
)

modelo_inicial = KeccakMILPModel(config_inicial)


print("=" * 70)
print("MODELO INICIAL")
print("=" * 70)
print(f"z                       : {config_inicial.z}")
print(f"Rondas                  : {config_inicial.rounds}")
print(f"Bits por estado         : {config_inicial.state_bits}")
print(f"Estados de frontera     : {config_inicial.rounds + 1}")
print(f"Variables declaradas    : {modelo_inicial.declared_variable_count()}")
print(f"Variables conectadas    : {modelo_inicial.attached_variable_count()}")
print(f"Restricciones           : {modelo_inicial.constraint_count()}")
print("=" * 70)

MODELO INICIAL
z                       : 4
Rondas                  : 1
Bits por estado         : 100
Estados de frontera     : 2
Variables declaradas    : 200
Variables conectadas    : 0
Restricciones           : 0


## 5. Variables declaradas y variables conectadas

La implementación mantiene un diccionario con todas las variables creadas:

$$
\texttt{state[(r,x,y,k)]}.
$$

Sin embargo, PuLP solo incorpora una variable a la representación activa del problema cuando participa en:

- una restricción;
- la función objetivo.

Por eso se distinguen dos cantidades:

### Variables declaradas

Son todas las variables creadas por la implementación:

$$
N_{\mathrm{declaradas}}
=
25z(R+1).
$$

### Variables conectadas

Son las variables que actualmente participan en alguna expresión del problema:

$$
N_{\mathrm{conectadas}}
\leq
N_{\mathrm{declaradas}}.
$$

Cuando se incorporen las transformaciones de las rondas, los estados intermedios y finales quedarán conectados mediante restricciones.

In [4]:
# ============================================================
# INSPECCIÓN DE VARIABLES DECLARADAS
# ============================================================

primeros_indices = list(modelo_inicial.state.keys())[:10]

registros_variables = []

for indice in primeros_indices:
    variable = modelo_inicial.state[indice]

    r, x, y, k = indice

    registros_variables.append(
        {
            "ronda_frontera": r,
            "x": x,
            "y": y,
            "k": k,
            "nombre_variable": variable.name,
            "categoria": variable.cat,
            "limite_inferior": variable.lowBound,
            "limite_superior": variable.upBound,
        }
    )

df_variables_muestra = pd.DataFrame(registros_variables)

df_variables_muestra

,ronda_frontera,x,y,k,nombre_variable,categoria,limite_inferior,limite_superior
0,0,0,0,0,a_r0_x0_y0_k0,Integer,0,1
1,0,0,0,1,a_r0_x0_y0_k1,Integer,0,1
2,0,0,0,2,a_r0_x0_y0_k2,Integer,0,1
3,0,0,0,3,a_r0_x0_y0_k3,Integer,0,1
4,0,0,1,0,a_r0_x0_y1_k0,Integer,0,1
5,0,0,1,1,a_r0_x0_y1_k1,Integer,0,1
6,0,0,1,2,a_r0_x0_y1_k2,Integer,0,1
7,0,0,1,3,a_r0_x0_y1_k3,Integer,0,1
8,0,0,2,0,a_r0_x0_y2_k0,Integer,0,1
9,0,0,2,1,a_r0_x0_y2_k1,Integer,0,1


## 6. Validación de los nombres e índices

Cada variable sigue el formato:

```text
a_r{r}_x{x}_y{y}_k{k}

In [5]:
# ============================================================
# ACCESO A UNA VARIABLE ESPECÍFICA
# ============================================================

variable_ejemplo = modelo_inicial.state_variable(
    round_index=0,
    x=1,
    y=2,
    k=3,
)

print("=" * 70)
print("VARIABLE DE EJEMPLO")
print("=" * 70)
print(f"Nombre          : {variable_ejemplo.name}")
print(f"Categoría       : {variable_ejemplo.cat}")
print(f"Límite inferior: {variable_ejemplo.lowBound}")
print(f"Límite superior: {variable_ejemplo.upBound}")
print("=" * 70)

assert variable_ejemplo.name == "a_r0_x1_y2_k3"

VARIABLE DE EJEMPLO
Nombre          : a_r0_x1_y2_k3
Categoría       : Integer
Límite inferior: 0
Límite superior: 1


## 7. Restricción de entrada diferencial no nula

Un modelo diferencial sin restricciones admitiría la solución trivial:

$$
A^{(0)}[x,y,k]=0
\qquad
\forall x,y,k.
$$

Esta solución tendría cero bits activos y, posteriormente, cero S-boxes activas. Sin embargo, no representa una diferencia criptográfica útil.

Por ello se impone:

$$
\sum_{x=0}^{4}
\sum_{y=0}^{4}
\sum_{k=0}^{z-1}
A^{(0)}[x,y,k]
\geq 1.
$$

Esta restricción garantiza que la diferencia inicial contenga al menos un bit activo.

In [6]:
# ============================================================
# AGREGAR RESTRICCIÓN DE ENTRADA NO NULA
# ============================================================

print("Antes de agregar la restricción:")
print(f"Variables conectadas: {modelo_inicial.attached_variable_count()}")
print(f"Restricciones       : {modelo_inicial.constraint_count()}")

modelo_inicial.add_nonzero_input_constraint()

print()
print("Después de agregar la restricción:")
print(f"Variables conectadas: {modelo_inicial.attached_variable_count()}")
print(f"Restricciones       : {modelo_inicial.constraint_count()}")
print(
    "Restricción encontrada:",
    modelo_inicial.has_constraint("entrada_diferencial_no_nula"),
)

Antes de agregar la restricción:
Variables conectadas: 0
Restricciones       : 0

Después de agregar la restricción:
Variables conectadas: 100
Restricciones       : 1
Restricción encontrada: True


## 8. Función objetivo provisional

La función objetivo definitiva será minimizar el número de S-boxes activas en las capas $\chi$.

Sin embargo, esas variables todavía no han sido creadas. Para validar la comunicación con el solver se utiliza provisionalmente:

$$
\min
\sum_{x=0}^{4}
\sum_{y=0}^{4}
\sum_{k=0}^{z-1}
A^{(0)}[x,y,k].
$$

Al combinar esta función con la restricción:

$$
\sum_{x,y,k} A^{(0)}[x,y,k]\geq1,
$$

el óptimo esperado es:

$$
Z^*=1.
$$

Esto significa que el solver elegirá exactamente una posición activa en el estado inicial.

In [7]:
# ============================================================
# AGREGAR OBJETIVO PROVISIONAL
# ============================================================

modelo_inicial.set_smoke_test_objective()

print("=" * 70)
print("OBJETIVO PROVISIONAL")
print("=" * 70)
print(modelo_inicial.problem.objective)
print("=" * 70)

OBJETIVO PROVISIONAL
a_r0_x0_y0_k0 + a_r0_x0_y0_k1 + a_r0_x0_y0_k2 + a_r0_x0_y0_k3 + a_r0_x0_y1_k0 + a_r0_x0_y1_k1 + a_r0_x0_y1_k2 + a_r0_x0_y1_k3 + a_r0_x0_y2_k0 + a_r0_x0_y2_k1 + a_r0_x0_y2_k2 + a_r0_x0_y2_k3 + a_r0_x0_y3_k0 + a_r0_x0_y3_k1 + a_r0_x0_y3_k2 + a_r0_x0_y3_k3 + a_r0_x0_y4_k0 + a_r0_x0_y4_k1 + a_r0_x0_y4_k2 + a_r0_x0_y4_k3 + a_r0_x1_y0_k0 + a_r0_x1_y0_k1 + a_r0_x1_y0_k2 + a_r0_x1_y0_k3 + a_r0_x1_y1_k0 + a_r0_x1_y1_k1 + a_r0_x1_y1_k2 + a_r0_x1_y1_k3 + a_r0_x1_y2_k0 + a_r0_x1_y2_k1 + a_r0_x1_y2_k2 + a_r0_x1_y2_k3 + a_r0_x1_y3_k0 + a_r0_x1_y3_k1 + a_r0_x1_y3_k2 + a_r0_x1_y3_k3 + a_r0_x1_y4_k0 + a_r0_x1_y4_k1 + a_r0_x1_y4_k2 + a_r0_x1_y4_k3 + a_r0_x2_y0_k0 + a_r0_x2_y0_k1 + a_r0_x2_y0_k2 + a_r0_x2_y0_k3 + a_r0_x2_y1_k0 + a_r0_x2_y1_k1 + a_r0_x2_y1_k2 + a_r0_x2_y1_k3 + a_r0_x2_y2_k0 + a_r0_x2_y2_k1 + a_r0_x2_y2_k2 + a_r0_x2_y2_k3 + a_r0_x2_y3_k0 + a_r0_x2_y3_k1 + a_r0_x2_y3_k2 + a_r0_x2_y3_k3 + a_r0_x2_y4_k0 + a_r0_x2_y4_k1 + a_r0_x2_y4_k2 + a_r0_x2_y4_k3 + a_r0_x3_y0_k0 + a_r

## 9. Estadísticas antes de resolver

El modelo inicial debería contener:

$$
N_{\mathrm{declaradas}}=200,
$$

$$
N_{\mathrm{conectadas}}=100,
$$

$$
N_{\mathrm{restricciones}}=1.
$$

Las 100 variables del estado de salida todavía no participan en ninguna expresión. Se conectarán cuando se implemente la primera ronda.

In [8]:
# ============================================================
# ESTADÍSTICAS ANTES DE RESOLVER
# ============================================================

estadisticas_antes = modelo_inicial.statistics()

df_estadisticas_antes = pd.DataFrame(
    [
        {
            "z": estadisticas_antes.z,
            "rondas": estadisticas_antes.rounds,
            "bits_estado": estadisticas_antes.state_bits,
            "estados_frontera": estadisticas_antes.boundary_states,
            "variables_declaradas": estadisticas_antes.declared_variables,
            "variables_conectadas": estadisticas_antes.attached_variables,
            "restricciones": estadisticas_antes.total_constraints,
        }
    ]
)

df_estadisticas_antes

,z,rondas,bits_estado,estados_frontera,variables_declaradas,variables_conectadas,restricciones
0,4,1,100,2,200,100,1


## 10. Resolución mediante CBC

El problema provisional se resuelve con CBC.

Se espera obtener:

$$
\text{estado}=\text{Optimal},
$$

y:

$$
Z^*=1.
$$

También se recuperará la posición elegida por el solver como diferencia inicial activa.

In [9]:
# ============================================================
# RESOLUCIÓN DEL MODELO INICIAL
# ============================================================

estado_solver = modelo_inicial.solve()
valor_objetivo = modelo_inicial.objective_value()
posiciones_activas = modelo_inicial.active_initial_positions()


print("=" * 70)
print("RESULTADO DEL MODELO INICIAL")
print("=" * 70)
print(f"Estado del solver     : {estado_solver}")
print(f"Valor objetivo        : {valor_objetivo}")
print(f"Posiciones activas    : {posiciones_activas}")
print(f"Número de posiciones : {len(posiciones_activas)}")
print("=" * 70)


assert estado_solver == "Optimal"
assert valor_objetivo == 1.0
assert len(posiciones_activas) == 1

RESULTADO DEL MODELO INICIAL
Estado del solver     : Optimal
Valor objetivo        : 1.0
Posiciones activas    : [(1, 1, 2)]
Número de posiciones : 1


## 11. Interpretación de la solución provisional

El solver encuentra una solución con un único bit activo porque:

$$
\sum A^{(0)}[x,y,k]\geq1,
$$

y simultáneamente minimiza:

$$
\sum A^{(0)}[x,y,k].
$$

Por tanto:

$$
\sum A^{(0)}[x,y,k]=1.
$$

Esta solución no representa todavía una trayectoria diferencial de Keccak, porque:

- el estado inicial no está conectado al estado final;
- no se ha aplicado $\theta$;
- no se ha aplicado $\rho$;
- no se ha aplicado $\pi$;
- no se ha aplicado $\chi$.

Su finalidad es únicamente validar la creación, construcción y resolución del problema MILP.

## 12. Construcción automática del esqueleto

El método `build_skeleton()` agrupa:

1. la restricción de entrada diferencial no nula;
2. la función objetivo provisional.

La operación es idempotente: ejecutarla más de una vez no debe duplicar restricciones ni objetivos.

In [10]:
# ============================================================
# VALIDACIÓN DE BUILD_SKELETON
# ============================================================

config_automatica = ExperimentConfig(
    z=4,
    rounds=2,
    solver="cbc",
    time_limit_seconds=60,
    mip_gap=0.0,
    verbose=False,
)

modelo_automatico = KeccakMILPModel(config_automatica)

modelo_automatico.build_skeleton()
restricciones_primera_ejecucion = modelo_automatico.constraint_count()

modelo_automatico.build_skeleton()
restricciones_segunda_ejecucion = modelo_automatico.constraint_count()


print(f"Restricciones después de la primera ejecución: "
      f"{restricciones_primera_ejecucion}")

print(f"Restricciones después de la segunda ejecución: "
      f"{restricciones_segunda_ejecucion}")


assert restricciones_primera_ejecucion == 1
assert restricciones_segunda_ejecucion == 1

Restricciones después de la primera ejecución: 1
Restricciones después de la segunda ejecución: 1


## 13. Comparación de las seis configuraciones

El trabajo contempla:

- $z=4$ y $z=8$;
- una, dos y tres rondas.

Para cada configuración, la cantidad de variables de estado es:

$$
N_{\mathrm{estado}}
=
25z(R+1).
$$

La siguiente tabla construye y resuelve el esqueleto para los seis escenarios.

Aunque los estados posteriores todavía no estén conectados, esta prueba permite comprobar:

- el crecimiento de las variables declaradas;
- el número de variables inicialmente conectadas;
- la capacidad del solver para procesar cada configuración;
- la consistencia del objetivo provisional.

In [11]:
# ============================================================
# EJECUCIÓN DE LOS SEIS ESQUELETOS
# ============================================================

resultados_esqueleto = []

for z in (4, 8):
    for rondas in (1, 2, 3):
        config = ExperimentConfig(
            z=z,
            rounds=rondas,
            solver="cbc",
            time_limit_seconds=60,
            mip_gap=0.0,
            verbose=False,
        )

        modelo = KeccakMILPModel(config)
        modelo.build_skeleton()

        estado = modelo.solve()
        estadisticas = modelo.statistics()

        resultados_esqueleto.append(
            {
                "z": z,
                "rondas": rondas,
                "bits_estado": config.state_bits,
                "estados_frontera": rondas + 1,
                "variables_declaradas": (
                    estadisticas.declared_variables
                ),
                "variables_conectadas": (
                    estadisticas.attached_variables
                ),
                "restricciones": (
                    estadisticas.total_constraints
                ),
                "estado_solver": estado,
                "objetivo": modelo.objective_value(),
                "bits_iniciales_activos": len(
                    modelo.active_initial_positions()
                ),
            }
        )


df_resultados_esqueleto = pd.DataFrame(
    resultados_esqueleto
)

df_resultados_esqueleto

,z,rondas,bits_estado,estados_frontera,variables_declaradas,variables_conectadas,restricciones,estado_solver,objetivo,bits_iniciales_activos
0,4,1,100,2,200,100,1,Optimal,1.0,1
1,4,2,100,3,300,100,1,Optimal,1.0,1
2,4,3,100,4,400,100,1,Optimal,1.0,1
3,8,1,200,2,400,200,1,Optimal,1.0,1
4,8,2,200,3,600,200,1,Optimal,1.0,1
5,8,3,200,4,800,200,1,Optimal,1.0,1


## 14. Validación matemática de los conteos

Para cada configuración debe cumplirse:

$$
N_{\mathrm{declaradas}}
=
25z(R+1),
$$

mientras que, en el esqueleto actual:

$$
N_{\mathrm{conectadas}}
=
25z.
$$

Esto ocurre porque solo el estado inicial participa en la restricción y en el objetivo provisional.

Una vez implementadas las rondas, las variables de los estados posteriores deberán conectarse mediante las restricciones de propagación.

In [12]:
# ============================================================
# VALIDACIONES DE LOS CONTEOS
# ============================================================

for _, fila in df_resultados_esqueleto.iterrows():
    z = int(fila["z"])
    rondas = int(fila["rondas"])

    esperado_declaradas = 25 * z * (rondas + 1)
    esperado_conectadas = 25 * z

    assert fila["variables_declaradas"] == esperado_declaradas
    assert fila["variables_conectadas"] == esperado_conectadas
    assert fila["restricciones"] == 1
    assert fila["estado_solver"] == "Optimal"
    assert fila["objetivo"] == 1.0
    assert fila["bits_iniciales_activos"] == 1


print("Los seis esqueletos fueron validados correctamente.")

Los seis esqueletos fueron validados correctamente.


## 15. Exportación de los resultados

Los resultados estructurales se guardan en:

```text
results/tables/resultados_esqueleto_milp.csv
```
Este archivo permitirá documentar el crecimiento inicial del modelo y mantener trazabilidad de las pruebas realizadas.

In [13]:
# ============================================================
# EXPORTACIÓN DE RESULTADOS
# ============================================================

ruta_resultados = (
    TABLES_DIR / "resultados_esqueleto_milp.csv"
)

df_resultados_esqueleto.to_csv(
    ruta_resultados,
    index=False,
    encoding="utf-8-sig",
)

print("=" * 70)
print("EXPORTACIÓN COMPLETADA")
print("=" * 70)
print(f"Archivo: {ruta_resultados}")
print(f"Existe : {ruta_resultados.exists()}")
print("=" * 70)

assert ruta_resultados.exists()

EXPORTACIÓN COMPLETADA
Archivo: d:\Documentos\000. MSC\3er Ciclo\Cripto\PracticaCalificada\results\tables\resultados_esqueleto_milp.csv
Existe : True


## 16. Estado actual del modelo

La estructura disponible es:

```text
Estado inicial A^(0)
        |
        | Entrada no nula
        |
        | Objetivo provisional
        v
      Solver
      La estructura que se construirá progresivamente será:
      A^(0)
        |
        v
        theta
        |
        v
        rho
        |
        v
        pi
        |
        v
        chi
        |
        v
        A^(1)
        |
        v
siguiente ronda

## 17. Conclusiones de la etapa

Se comprobó que:

1. el modelo crea correctamente los estados de frontera;
2. la cantidad de variables declaradas cumple:

$$
N_{\mathrm{estado}}
=
25z(R+1);
$$

3. PuLP distingue entre variables declaradas y variables conectadas;
4. la entrada diferencial nula fue excluida;
5. el objetivo provisional produce un óptimo igual a uno;
6. CBC resuelve correctamente las seis configuraciones;
7. la estructura está preparada para incorporar las capas de las rondas.

La siguiente etapa consistirá en implementar y validar la transformación $\theta$.

Esta capa requiere modelar operaciones XOR, por lo que será necesario introducir variables auxiliares de paridad y restricciones lineales adicionales.

In [14]:
# ============================================================
# CONTROL FINAL
# ============================================================

assert len(df_resultados_esqueleto) == 6
assert (
    df_resultados_esqueleto["estado_solver"]
    == "Optimal"
).all()
assert (
    df_resultados_esqueleto["objetivo"]
    == 1.0
).all()
assert ruta_resultados.exists()

print("=" * 70)
print("ESQUELETO MILP VALIDADO")
print("=" * 70)
print("El proyecto está listo para modelar la capa theta.")
print("=" * 70)

ESQUELETO MILP VALIDADO
El proyecto está listo para modelar la capa theta.
